# **Special Methods and Operator Overloading**

**DS212 — Object-Oriented Programming**

**Week 10:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Implement special methods and operator overloading to define meaningful object behavior.

Why does `len()` work on a list, a string, **and** a pandas DataFrame? Why can we write `a + b` for numbers, strings, **and** NumPy arrays?

The answer is the **Python data model**: a set of **special methods** (also called *magic* or *dunder* methods, because their names have **d**ouble **under**scores) that Python calls automatically when you use built-in syntax.

```text
What you write          What Python calls
──────────────          ─────────────────
len(x)            →     x.__len__()
str(x)            →     x.__str__()
a + b             →     a.__add__(b)
a == b            →     a.__eq__(b)
x[i]              →     x.__getitem__(i)
for item in x     →     x.__iter__()
```

By implementing these methods in **our own classes**, our objects can work naturally with Python's syntax and built-in functions. This is a form of **polymorphism**: the same operator behaves appropriately for each type.

In [ ]:
# the same special methods are behind familiar built-in behavior
numbers = [3, 1, 2]
print(len(numbers), numbers.__len__())
print(3 + 4, (3).__add__(4))
print("DS" + "212", "DS".__add__("212"))

## **Object Representation**

When you print an object of a new class, Python shows something unhelpful:

In [ ]:
class Student:
    def __init__(self, name, gwa):
        self.name = name
        self.gwa = gwa


print(Student("Ana", 1.25))

Two special methods control how an object is shown:

| Method       | Called by                         | Audience        | Goal                                       |
| ------------ | --------------------------------- | --------------- | ------------------------------------------ |
| `__repr__()` | `repr(obj)`, the notebook, lists  | **developers**  | unambiguous; ideally looks like code that recreates the object |
| `__str__()`  | `str(obj)`, `print(obj)`, f-strings | **end users** | readable                                   |

If only `__repr__()` is defined, Python uses it for `str()` as well — so **always define `__repr__()` first**.

In [ ]:
class Student:
    def __init__(self, name, gwa):
        self.name = name
        self.gwa = gwa

    def __repr__(self):
        return f"Student(name={self.name!r}, gwa={self.gwa!r})"

    def __str__(self):
        return f"{self.name} (GWA {self.gwa:.2f})"


ana = Student("Ana", 1.25)
print(ana)              # __str__
print(repr(ana))        # __repr__
print([ana])            # containers use __repr__ for their items
ana                     # the notebook displays __repr__

### **Custom formatting with `__format__`**

`__format__()` lets an object respond to format specifications in f-strings:

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __format__(self, spec):
        if spec == "php":
            return f"₱{self.amount:,.2f}"
        return format(self.amount, spec)


fee = Money(18250)
print(f"Tuition: {fee:php}")
print(f"Raw: {fee:.1f}")

## **Lifecycle Methods**

Special methods also take part in an object's **life cycle**.

```{mermaid}
flowchart LR
    A["ClassName(...)"] --> B["__new__()<br/>create the object"]
    B --> C["__init__()<br/>initialize state"]
    C --> D["object in use"]
    D --> E["no references left"]
    E --> F["__del__()<br/>finalize (rarely needed)"]
```

* `__new__()` **creates** the object — rarely overridden, except for immutable types or special patterns.
* `__init__()` **initializes** it — the constructor you already know.
* `__del__()` runs when the object is about to be destroyed. Its timing is **not guaranteed**, so it should not be used for important cleanup.

For reliable cleanup — closing files, database connections, timers — Python uses **context managers**, defined with `__enter__()` and `__exit__()` and used through the `with` statement:

In [ ]:
import time


class Timer:
    def __init__(self, label):
        self.label = label

    def __enter__(self):                     # runs at the start of the with-block
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):   # ALWAYS runs at the end, even on error
        elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] finished in {elapsed:.4f} s")
        return False                         # do not hide exceptions


with Timer("sum of squares"):
    total = sum(i * i for i in range(200_000))

print(total)

## **Comparison Special Methods**

By default, `==` compares **identity**: two objects are equal only if they are the same object.

In [ ]:
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor


print(Version(1, 2) == Version(1, 2))

Comparison operators map to these methods:

| Operator | Method      | | Operator | Method      |
| -------- | ----------- |-| -------- | ----------- |
| `==`     | `__eq__`    | | `<`      | `__lt__`    |
| `!=`     | `__ne__`    | | `<=`     | `__le__`    |
| `>`      | `__gt__`    | | `>=`     | `__ge__`    |

Writing all six is tedious. `functools.total_ordering` generates the rest from `__eq__` and **one** ordering method:

In [ ]:
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor

    def _key(self):
        return (self.major, self.minor)

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented            # let Python try other options
        return self._key() == other._key()

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._key() < other._key()

    def __hash__(self):                      # equal objects must have equal hashes
        return hash(self._key())

    def __repr__(self):
        return f"v{self.major}.{self.minor}"


releases = [Version(1, 10), Version(1, 2), Version(0, 9)]
print(Version(1, 2) == Version(1, 2))
print(Version(1, 10) > Version(1, 2))       # generated by total_ordering
print(sorted(releases), max(releases))
print({Version(1, 2), Version(1, 2)})       # usable in sets because of __hash__

> **Important:** if you define `__eq__()`, Python sets `__hash__` to `None`, making objects **unhashable** (they cannot be set items or dictionary keys) unless you also define `__hash__()`. Only hash objects whose compared values **do not change**.

Returning `NotImplemented` (not raising an error) tells Python: *"I don't know how to compare with that type."* Python then tries the other operand's method, and finally falls back to a sensible default.

## **Arithmetic Special Methods**

| Operator | Method        | Reflected (right side) | In-place      |
| -------- | ------------- | ---------------------- | ------------- |
| `+`      | `__add__`     | `__radd__`             | `__iadd__`    |
| `-`      | `__sub__`     | `__rsub__`             | `__isub__`    |
| `*`      | `__mul__`     | `__rmul__`             | `__imul__`    |
| `/`      | `__truediv__` | `__rtruediv__`         | `__itruediv__`|
| `-x`     | `__neg__`     |                        |               |
| `abs(x)` | `__abs__`     |                        |               |

Consider a 2-D vector, common in data science and graphics:

```{mermaid}
classDiagram
class Vector {
    +float x
    +float y
    +__add__(other) Vector
    +__sub__(other) Vector
    +__mul__(scalar) Vector
    +__rmul__(scalar) Vector
    +__abs__() float
    +__eq__(other) bool
    +__repr__() str
}
```

In [ ]:
import math


class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):               # vector * number
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):              # number * vector
        return self * scalar

    def __neg__(self):
        return Vector(-self.x, -self.y)

    def __abs__(self):                       # magnitude
        return math.hypot(self.x, self.y)

    def __bool__(self):                      # a zero vector is "falsy"
        return bool(abs(self))


a = Vector(3, 4)
b = Vector(1, 2)
print(a + b, a - b, a * 2, 3 * b, -a)
print(abs(a), bool(Vector(0, 0)))

try:
    a + 5
except TypeError as error:
    print("TypeError:", error)

The last line shows why `NotImplemented` matters: `Vector + int` is meaningless, so Python raises a clear `TypeError` instead of producing nonsense.

**Reflected methods** such as `__rmul__()` handle the case where our object is on the **right**: for `3 * b`, Python first tries `int.__mul__(3, b)`, which returns `NotImplemented`, and then calls `b.__rmul__(3)`.

`sum()` starts from `0`, so to make `sum(vectors)` work you can give a start value: `sum(vectors, Vector(0, 0))`.

## **Container Special Methods**

Objects that hold collections of items can behave like built-in containers:

| Syntax             | Method           |
| ------------------ | ---------------- |
| `len(obj)`         | `__len__`        |
| `obj[key]`         | `__getitem__`    |
| `obj[key] = value` | `__setitem__`    |
| `item in obj`      | `__contains__`   |
| `for x in obj`     | `__iter__`       |
| `obj(...)`         | `__call__`       |

A small `Dataset` class shows how a data science object can feel as natural as a list:

In [ ]:
class Dataset:
    def __init__(self, name, rows):
        self.name = name
        self._rows = list(rows)

    def __repr__(self):
        return f"Dataset({self.name!r}, {len(self)} rows)"

    def __len__(self):
        return len(self._rows)

    def __getitem__(self, index):
        if isinstance(index, slice):                  # support ds[1:3]
            return Dataset(self.name, self._rows[index])
        return self._rows[index]

    def __iter__(self):
        return iter(self._rows)

    def __contains__(self, name):
        return any(row["name"] == name for row in self._rows)

    def __call__(self, column):                       # ds("score") → that column
        return [row[column] for row in self._rows]


grades = Dataset("DS212", [
    {"name": "Ana", "score": 92},
    {"name": "Ben", "score": 85},
    {"name": "Carlo", "score": 78},
])

print(grades, len(grades))
print(grades[0], grades[1:])
print("Ben" in grades, "Zoe" in grades)
print([row["name"] for row in grades])
print(grades("score"), max(grades("score")))

## **Shortcut: Data Classes**

Many classes need the same boilerplate: `__init__`, `__repr__`, and `__eq__`. The `@dataclass` decorator **generates** these special methods from type-annotated fields:

In [ ]:
from dataclasses import dataclass, field


@dataclass(order=True)
class Grade:
    score: float
    student: str = field(compare=False)
    remarks: str = field(default="", compare=False)


g1 = Grade(92, "Ana")
g2 = Grade(85, "Ben", "late")
print(g1)                            # generated __repr__
print(g1 == Grade(92, "Someone"))    # generated __eq__ compares only 'score'
print(sorted([g1, g2]))              # generated ordering methods

## **Appropriate Use of Operator Overloading**

Operator overloading is powerful, but it can also make code **confusing**. Follow these guidelines:

| Do                                                                    | Avoid                                                                 |
| --------------------------------------------------------------------- | --------------------------------------------------------------------- |
| Overload when the operator's **usual meaning** clearly applies (`Vector + Vector`, `Money + Money`) | Surprising meanings (`student + course` to enroll)    |
| Keep operations **consistent**: if `a + b` works, `b + a` should usually too | Operators with hidden side effects (`a + b` modifying `a`)     |
| Return **new objects** from `+`, `-`, `*`                              | Raising errors for unsupported types — return `NotImplemented`        |
| Always define `__repr__()`                                            | Defining `__eq__()` without thinking about `__hash__()`               |
| Prefer a **named method** when the meaning is not obvious (`course.enroll(student)`) | Overloading just because it is possible                 |

> **Rule of thumb:** a reader who has never seen your class should be able to **guess** what an operator does. If they cannot, use a method name instead.

## **Summary**

| Group           | Special methods                                         | Enables                                  |
| --------------- | ------------------------------------------------------- | ---------------------------------------- |
| Representation  | `__repr__`, `__str__`, `__format__`                     | readable output and debugging            |
| Lifecycle       | `__new__`, `__init__`, `__del__`, `__enter__`, `__exit__` | creation, initialization, cleanup     |
| Comparison      | `__eq__`, `__lt__`, … , `__hash__`                      | `==`, `<`, `sorted()`, sets, dict keys    |
| Arithmetic      | `__add__`, `__radd__`, `__mul__`, `__neg__`, `__abs__`  | natural math on objects                  |
| Containers      | `__len__`, `__getitem__`, `__iter__`, `__contains__`    | `len()`, indexing, loops, `in`           |
| Callables       | `__call__`                                              | objects used like functions              |

## **References**

Downey, A. B. (2024). *Think Python: How to think like a computer scientist* (3rd ed.). O'Reilly Media. https://allendowney.github.io/ThinkPython/

Lott, S. F., & Phillips, D. (2021). *Python object-oriented programming* (4th ed.). Packt Publishing.

Python Software Foundation. (2026). *dataclasses — Data classes*. https://docs.python.org/3/library/dataclasses.html

Python Software Foundation. (2026). *functools — Higher-order functions and operations on callable objects*. https://docs.python.org/3/library/functools.html

Python Software Foundation. (2026). *The Python language reference: 3. Data model*. https://docs.python.org/3/reference/datamodel.html

Python Software Foundation. (2026). *The Python tutorial: 9. Classes*. https://docs.python.org/3/tutorial/classes.html

Ramalho, L. (2022). *Fluent Python: Clear, concise, and effective programming* (2nd ed.). O'Reilly Media.